# Train NLQT-Qwen3-1.7B-v2: SciX NL Query Translator

Fine-tunes Qwen3-1.7B on cleaned NL-to-ADS-query training data.

**What's new in v2:**
- Cleaned training data: normalized year syntax, fixed negation parsing, removed synonym expansion examples
- Supports two output formats: query string (default) or IntentSpec JSON with `<think>` reasoning
- Validation set evaluation during training

**Requirements:**
- Google Colab with T4 GPU (free tier works)
- ~2-3 hours for training (~3,700 pairs)
- HuggingFace account for upload (optional)

**Output:**
- Merged model ready for deployment or upload to HuggingFace

## 1. Setup Environment

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install dependencies
!pip install -q torch transformers>=4.45.0 datasets peft accelerate trl
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Clone Repo and Build Dataset

In [ ]:
# Use local repo (no clone needed when running Colab locally)
import os, sys
os.chdir("/Users/mugdhapolimera/github/nls-finetune-scix")
sys.path.insert(0, "packages/finetune/src")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Rebuild the processed dataset
# This validates, deduplicates, and splits into train/val
#
# Two format options:
#   "intent" — Phase B: <think> + IntentSpec JSON (recommended)
#   "query"  — Legacy: {"query": "..."}  query string format

OUTPUT_FORMAT = "intent"  # Change to "query" for legacy format

# For intent format, first convert gold_examples.json → gold_examples_intent.json
# (adds intent_json and think_trace fields via parse_query round-trip)
if OUTPUT_FORMAT == "intent":
    !python scripts/convert_to_intent_format.py \
        --input data/datasets/raw/gold_examples.json \
        --output data/datasets/raw/gold_examples_intent.json

# Build train/val splits
# --gold-only skips synthetic/NL pairs that don't have intent_json
GOLD_FILE = "data/datasets/raw/gold_examples_intent.json" if OUTPUT_FORMAT == "intent" else "data/datasets/raw/gold_examples.json"
GOLD_ONLY = "--gold-only" if OUTPUT_FORMAT == "intent" else ""
!python scripts/build_dataset.py \
    --gold {GOLD_FILE} \
    {GOLD_ONLY} \
    --output-dir data/datasets/processed \
    --format {OUTPUT_FORMAT} \
    --date 2026-03-17 \
    --seed 42

## 3. Configuration

In [ ]:
import json
import torch

# === Model ===
MODEL_NAME = "unsloth/Qwen3-1.7B"       # Unsloth-optimized Qwen3-1.7B
BASE_MODEL = "Qwen/Qwen3-1.7B"          # For merge step
OUTPUT_DIR = "./output/NLQT-Qwen3-1.7B-v2"

# === Data ===
TRAIN_FILE = "data/datasets/processed/train.jsonl"
VAL_FILE = "data/datasets/processed/val.jsonl"

# === LoRA ===
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# === Training (T4-optimized) ===
MAX_SEQ_LENGTH = 512
EPOCHS = 3
BATCH_SIZE = 2                           # T4 has 16GB — use 2 with 4-bit quant
GRADIENT_ACCUMULATION_STEPS = 8          # Effective batch size = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
SEED = 42

# T4 = fp16 only. A100/H100 = bf16.
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
USE_BF16 = "A100" in gpu_name or "H100" in gpu_name
USE_FP16 = not USE_BF16

print(f"GPU: {gpu_name}")
print(f"Precision: {'bf16' if USE_BF16 else 'fp16'}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Output: {OUTPUT_DIR}")

## 4. Load Data

In [ ]:
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

train_data = load_jsonl(TRAIN_FILE)
val_data = load_jsonl(VAL_FILE)

print(f"Train: {len(train_data)} examples")
print(f"Val:   {len(val_data)} examples")
print(f"\nSystem prompt (first 120 chars):")
print(f"  {train_data[0]['messages'][0]['content'][:120]}...")
print(f"\nExample input:")
print(f"  {train_data[0]['messages'][1]['content']}")
print(f"\nExample output (first 200 chars):")
print(f"  {train_data[0]['messages'][2]['content'][:200]}")

## 5. Load Model + Apply LoRA

In [ ]:
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
from datasets import Dataset

# Load model — use fp16 on T4, bf16 on A100
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    load_in_4bit=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)")

In [ ]:
# Prepare datasets: apply chat template
def format_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_data)
train_dataset = train_dataset.map(format_chat_template, remove_columns=train_dataset.column_names)

val_dataset = Dataset.from_list(val_data)
val_dataset = val_dataset.map(format_chat_template, remove_columns=val_dataset.column_names)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset:   {len(val_dataset)} examples")
print(f"\nSample formatted text (first 300 chars):")
print(train_dataset[0]["text"][:300])

## 6. Train

In [ ]:
training_args = UnslothTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    bf16=USE_BF16,
    fp16=USE_FP16,
    optim="adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print(f"Training config:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Precision: {'bf16' if USE_BF16 else 'fp16'}")
print(f"\nStarting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Save LoRA adapter
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

## 7. Merge LoRA into Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Loading base model for merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, f"{OUTPUT_DIR}/lora_adapter")

print("Merging...")
merged_model = model.merge_and_unload()

MERGED_DIR = f"{OUTPUT_DIR}/merged"
print(f"Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
base_tokenizer.save_pretrained(MERGED_DIR)

print("Merge complete!")

## 8. Quick Eval: Test 10 Queries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading merged model for testing...")
test_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
test_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)

# Use the same system prompt as training
system_prompt = train_data[0]["messages"][0]["content"]

test_queries = [
    "papers about exoplanets published in 2023",
    "articles by John Smith on machine learning",
    "highly cited papers about dark matter",
    "recent gravitational wave detections in Nature",
    "Hawking radiation papers from the 1970s",
    "first author Tenenbaum on stellar oscillations",
    "papers NOT about cosmology by Perlmutter",
    "JWST observations of early galaxies 2022 to 2024",
    "refereed reviews about fast radio bursts",
    "papers citing the original LIGO detection paper",
]

print(f"System prompt: {system_prompt[:80]}...\n")
print("=" * 70)

for query in test_queries:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Query: {query}\nDate: 2026-03-17"},
    ]
    prompt = test_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = test_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=test_tokenizer.eos_token_id,
        )

    response = test_tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Input:  {query}")
    print(f"Output: {response}")
    print("-" * 70)

## 9. Save / Push to HuggingFace

In [ ]:
# Optional: Login to HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Upload merged model to HuggingFace
REPO_ID = "adsabs/NLQT-Qwen3-1.7B-v2"  # Change as needed

merged_model.push_to_hub(REPO_ID, safe_serialization=True)
base_tokenizer.push_to_hub(REPO_ID)

print(f"\nModel uploaded to: https://huggingface.co/{REPO_ID}")

In [ ]:
# Alternative: Download merged model as zip
from google.colab import files
!zip -r merged_model.zip {MERGED_DIR}
files.download('merged_model.zip')

## Done!

### Deployment

**vLLM (recommended):**
```bash
pip install vllm
vllm serve adsabs/NLQT-Qwen3-1.7B-v2 --max-model-len 512
```

**Docker (with the project server):**
```bash
cd docker
docker compose up
```